# Strategic Patient Risk Stratification & Readmission Predictive Modeling
### Vitality Health Network (VHN) — Clinical Decision Support & Quality Improvement

Deliverable: Technical Appendix & Exploratory Data Analysis Notebook.                                             
Role: Health Informatics Consulting Group.                                              
Target Stakeholders: Chief Medical Information Officer (CMIO), Chief Nursing Officer,       Director of Quality & Care Management. 

---

#### Executive & Pedagogical Context
Under the Hospital Readmissions Reduction Program (HRRP) established by the Centers for Medicare & Medicaid Services (CMS), healthcare delivery systems face substantial reimbursement penalties when 30-day all-cause readmission rates for chronic conditions exceed national risk-adjusted benchmarks. 

At Vitality Health Network (VHN), the historical 30-day readmission rate for diabetic patients has reached 18%, far outpacing benchmark expectations and incurring millions of dollars in annual penalty exposure. VHN executive leadership has mandated the transition from reactive retrospective reporting to proactive, diagnostic, and prescriptive risk stratification.

This notebook provides the complete analytical pipeline, structured across four rigorous phases:
1. Phase 1: Data Ingestion & Clinical Sanitation.
2. Phase 2: Data Enrichment via Web Scraping.
3. Phase 3: Exploratory Data Analysis (EDA).
4. Phase 4: Feature Engineering — The Vitality Complexity Index (VCI).

## Phase 01: Data Ingestion & Clinical Sanitation

#### Project: Patient Risk Stratification – Vitality Health Network (VHN)

#### Phase Overview

This phase focuses on preparing the raw clinical dataset for reliable
data analysis and patient risk stratification.

The main objectives of this phase are:

1. Load the clinical dataset into a Pandas DataFrame.
2. Perform an initial data quality and structure assessment.
3. Standardize non-standard missing values.
4. Analyze missing data.
5. Apply ID mappings to obtain meaningful descriptions.
6. Exclude encounters associated with expired patients.
7. Remove exact duplicate records while preserving valid patient encounters.

#### Expected Outcome

A clean and structured encounter-level dataset that can be
used for the subsequent analytical phases.

In [ ]:
import time
import re
import warnings
from pathlib import Path
import csv

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import requests
from bs4 import BeautifulSoup

import matplotlib.pyplot as plt
import seaborn as sns


print("\nRequired libraries imported successfully.")

### 1.1 Loading the Clinical Dataset

The `diabetic_data.csv` file contains patient encounter information
collected from the hospital's electronic health record system.

The dataset is loaded into a Pandas DataFrame for further processing.

The original dataset uses the question mark (`?`) to represent
missing values in certain columns.

The `na_values` parameter converts these non-standard missing-value
markers into standard NumPy `NaN` values during data ingestion.

This ensures consistent missing-value identification throughout
the data cleaning process.

In [ ]:
# Step 1.2: Define the dataset location

data_dir = Path("../data_files")
raw_data_path = data_dir / "diabetic_data.csv"

if not raw_data_path.exists():
    raise FileNotFoundError(
        f"Dataset file not found: {raw_data_path.resolve()}"
    )
  

df_raw = pd.read_csv(
    raw_data_path,
    na_values=["?"],
    low_memory=False
)

print("Clinical dataset loaded successfully!")

### 1.2 Initial Data Assessment

An initial assessment is performed to understand the structure,
features, data types, and statistical characteristics of the dataset.

The following Pandas methods are used:

1. `shape` – Identifies the number of rows and columns.
2. `columns` – Displays all available feature names.
3. `info()` – Provides information about data types and non-null values.
4. `describe()` – Generates descriptive statistics for dataset features.
5. `head()` – Displays the first five records.

Special attention is given to identifier columns because their
numerical values represent categories rather than measurable quantities.

This assessment helps identify potential data quality issues
before applying further cleaning operations.

In [ ]:
# Step 1.2: Perform an initial assessment of the raw dataset

print("Dataset Dimensions:")
print(df_raw.shape)

print("\nDataset Column Names:")
print(df_raw.columns.tolist())

print("\nDataset Information:")
df_raw.info()

print("\nDescriptive Statistical Summary:")
display(df_raw.describe(include="all").T)

print("\nFirst Five Records:")
df_raw.head()

### 1.3 Loading the ID Mapping Dataset

The `IDs_mapping.csv` file contains descriptions for the
categorical identifier columns in the clinical dataset.

These mappings are required to interpret the numerical
identifiers correctly.

The following identifier columns are considered:

1. admission_type_id
2. discharge_disposition_id
3. admission_source_id

The mapping file is structured into separate sections.
Therefore, each section is extracted and stored as a
separate dictionary for further processing.

This approach improves the interpretability of the clinical
dataset and supports the identification of expired patients.

In [ ]:
# Step 1.3: Load and extract ID mappings

mapping_file = data_dir / "IDs_mapping.csv"


if not mapping_file.exists():
    raise FileNotFoundError(
        f"Mapping file not found: {mapping_file.resolve()}"
    )


with mapping_file.open(
    mode="r",
    encoding="utf-8-sig",
    newline=""
) as file:

    mapping_rows = list(csv.reader(file))


required_sections = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]


id_mappings = {}


for section in required_sections:

    mapping_dict = {}
    section_found = False

    for index, row in enumerate(mapping_rows):

        if row and row[0].strip().lower() == section:

            section_found = True

            for mapping_row in mapping_rows[index + 1:]:

                if not mapping_row or not mapping_row[0].strip():
                    continue

                first_value = mapping_row[0].strip().lower()

                if first_value in required_sections:
                    break

                if len(mapping_row) >= 2:
                    mapping_id = mapping_row[0].strip()
                    description = mapping_row[1].strip()

                    if mapping_id.isdigit():
                        mapping_dict[mapping_id] = description

            break

    if not section_found:
        raise ValueError(
            f"Mapping section '{section}' was not found."
        )

    id_mappings[section] = mapping_dict


for section, mapping_dict in id_mappings.items():

    print(f"\n{section}")
    print("-" * 50)

    for mapping_id, description in mapping_dict.items():
        print(f"{mapping_id}: {description}")

print("\nID mappings loaded successfully.")

### 1.4 Applying ID Mappings

The numerical identifiers in the clinical dataset represent
categorical information rather than continuous numerical values.

The extracted ID mappings are used to create additional
description columns for better data interpretation.

The following description columns are generated:

- admission_type_description
- discharge_disposition_description
- admission_source_description

The original identifier columns are preserved to maintain
the integrity of the raw clinical data.

These descriptions will also help identify encounters
associated with expired patients.

In [ ]:
# Step 1.4: Apply ID mappings to the clinical dataset


df_cleaned = df_raw.copy()


mapping_columns = {
    "admission_type_id": "admission_type_description",
    "discharge_disposition_id": "discharge_disposition_description",
    "admission_source_id": "admission_source_description"
}


for id_column, description_column in mapping_columns.items():

    id_values = pd.to_numeric(
        df_cleaned[id_column],
        errors="coerce"
    ).astype("Int64").astype("string")


    df_cleaned[description_column] = id_values.map(
        id_mappings[id_column]
    )


for id_column, description_column in mapping_columns.items():

    print(f"\n{id_column} and {description_column}")
    display(df_cleaned[[id_column, description_column]].drop_duplicates().head(15))
    

print("\nID descriptions applied successfully.")

### 1.5 Missing Value Analysis

Missing values can affect the accuracy and reliability
of clinical data analysis.

Although the question mark (`?`) was converted into
standard missing values during data ingestion, a detailed
missing value assessment is required.

This section calculates:

1. Total missing values in each column.
2. Percentage of missing values in each column.
3. Columns containing missing data.

The results are arranged in descending order based on
the percentage of missing values.

This analysis helps identify columns that require
special handling before further data processing.

In [ ]:
# Step 1.5: Analyze missing values in the dataset

missing_counts = df_cleaned.isnull().sum()

missing_percentages = (
    df_cleaned.isnull().mean() * 100
)


missing_summary = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing Percentage (%)": missing_percentages
})

missing_summary = missing_summary.sort_values(
    by="Missing Percentage (%)",
    ascending=False
)


print("Missing Value Analysis:")

display(
    missing_summary[
        missing_summary["Missing Count"] > 0
    ]
)


print(
    "\nTotal Missing Values:",
    df_cleaned.isnull().sum().sum()
)

### 1.6 Weight Feature Analysis

The `weight` column is examined to determine the extent
of missing information.

A missing value percentage greater than 90% indicates
that the feature contains insufficient information for
reliable analysis.

According to the project requirements, if more than 90%
of the values are missing, the entire `weight` column
must be removed.

This decision is documented as a Data Quality Limitation
because removing the feature results in the loss of
potentially useful clinical information.

If the missing percentage does not exceed 90%, the
column is retained for further consideration.

In [ ]:
# Step 1.6: Analyze and handle missing values in the weight column


if "weight" in df_cleaned.columns:

    weight_missing_percentage = (
        df_cleaned["weight"].isnull().mean() * 100
    )

    print("Weight Column Analysis")
    print("-" * 40)

    print(
        "Total Records:",
        len(df_cleaned)
    )

    print(
        "Missing Values:",
        df_cleaned["weight"].isnull().sum()
    )

    print(
        f"Missing Percentage: "
        f"{weight_missing_percentage:.2f}%"
    )

    if weight_missing_percentage > 90:

        df_cleaned.drop(
            columns=["weight"],
            inplace=True
        )

        print("\nData Quality Limitation:")
        print(
            "The weight column was removed because "
            "more than 90% of its values were missing."
        )

    else:

        print(
            "\nThe weight column is retained because "
            "its missing percentage does not exceed 90%."
        )

else:

    print(
        "The weight column does not exist in the dataset."
    )

### 1.7 Gender Data Sanitation & Demographic Validation

The `gender` feature represents a critical demographic attribute required for clinical risk stratification, epidemiological profiling, and health equity analysis.

An initial audit of the demographic distribution reveals non-standard entries categorized as `'Unknown/Invalid'`. These values lack clinical validity and demographic utility for downstream predictive modeling.

This step performs the following data sanitation tasks:

1. Inspect the frequency distribution of gender categories before cleaning.
2. Standardize non-valid demographic entries (`'Unknown/Invalid'`) into standard `NaN` values to ensure consistent missing-value tracking.
3. Validate the post-cleaning category distribution and verify the total count of missing values.

This process ensures that invalid entries do not distort demographic summaries, feature encodings, or disparity assessments in subsequent analytical phases.

In [ ]:
# Step 1.7: Standardize invalid entries in the gender column

print("Before cleaning:")
print(df_cleaned['gender'].value_counts(dropna=False))


df_cleaned['gender'] = df_cleaned['gender'].replace('Unknown/Invalid', np.nan)


print("\nAfter cleaning:")
print(df_cleaned['gender'].value_counts(dropna=False))


print("\nMissing gender values:", df_cleaned['gender'].isnull().sum())

### 1.8 Handling Deceased Patients

Patient encounters associated with expired patients must be
excluded before performing readmission analysis.

The `discharge_disposition_id` column contains identifiers
representing the patient's discharge destination or status.

The corresponding descriptions from `IDs_mapping.csv` are
used to identify discharge records associated with expired patients.

All discharge disposition descriptions containing the word
"Expired" are identified.

The corresponding patient encounters are removed from the
working dataset.

The original raw dataset is preserved, and the cleaning
operation is performed on `df_cleaned`.

This step helps prevent deceased patients from being included
in subsequent readmission analysis.

In [ ]:
# Step 1.7: Identify and filter deceased/terminal patients

expired_codes = [11, 19, 20, 21]

deceased_mask = df_cleaned['discharge_disposition_id'].isin(expired_codes)
num_deceased = deceased_mask.sum()
initial_count = len(df_cleaned)

df_cleaned = df_cleaned[~deceased_mask].copy()

records_after_removal = len(df_cleaned) 

print("\nExpired Patient Removal Summary")
print("-" * 45)

print("Records Before Removal:", initial_count)
print("Expired Encounters Removed:", num_deceased)
print("Records After Removal:", records_after_removal)


print("\nExpired patient encounters handled successfully.")

### 1.9 Removing Duplicate Records

Duplicate records can affect the accuracy and reliability
of clinical data analysis.

The dataset is examined to identify exact duplicate rows
using all available columns.

Only records that are completely identical are removed.

Multiple encounters belonging to the same patient are valid
clinical records and must be preserved.

Therefore, deduplication is performed using the entire
record rather than the patient identifier (`patient_nbr`).

The number of duplicate records is calculated before removal,
and the dataset is checked again after the cleaning process.

In [ ]:
# Step 1.9: Identify and remove exact duplicate records

duplicate_count = df_cleaned.duplicated().sum()

print("Duplicate Records Analysis")
print("-" * 40)

print("Records Before Deduplication:", len(df_cleaned))
print("Exact Duplicate Records Found:", duplicate_count)


df_cleaned = df_cleaned.drop_duplicates().copy()

print("\nDeduplication Summary")
print("-" * 40)

print("Records After Deduplication:", len(df_cleaned))

if duplicate_count > 0:
    df_cleaned.drop_duplicates(inplace=True)
    print("Exact duplicates dropped.")

unique_patients = df_cleaned['patient_nbr'].nunique()
print(f"Unique Individual Patients: {unique_patients:,}")

repeat_counts = df_cleaned['patient_nbr'].value_counts()
print(f"Patients with multiple hospitalizations: {(repeat_counts > 1).sum():,} ({(repeat_counts > 1).sum() / unique_patients * 100:.2f}%)")

print("\nExact duplicate records handled successfully.")

### 1.10 Schema Standardization & Categorical Type Conversion

During initial ingestion, columns containing numeric identifiers (such as admission_type_id, discharge_disposition_id, and admission_source_id) are parsed as integers (int64). 

Treating categorical codes as numeric measurements is clinically and statistically invalid:
1. Mathematical Inappropriateness: Arithmetic operations (such as computing mean or standard deviation on admission or discharge codes) are meaningless.
2. Algorithm Distortion: Machine learning algorithms would misinterpret these discrete categories as continuous numerical scales with intrinsic ordinal rank.
3. Demographic & Operational Attributes: Key clinical features (race, gender, age, payer_code, medical_specialty) represent discrete nominal/ordinal factors that should be formally defined as categories.

In this step, we:
* Enable full column visibility across the Pandas session.
* Convert numerical identifier columns and demographic features into the Pandas category data type.
* Verify the updated schema to ensure proper representation and optimized memory utilization.

In [ ]:
# Step 1.10: Convert identifier and demographic columns to categorical data types

print("Show all columns before converted")
print("-" * 50)
print(f"Total columns: {len(df_cleaned.columns)}\n")

with pd.option_context('display.max_rows', None):
    display(df_cleaned.dtypes.to_frame(name='Data Type'))


categorical_columns = [
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id',
    'race',
    'gender',
    'age',
    'payer_code',
    'medical_specialty'
]


for column in categorical_columns:
    df_cleaned[column] = df_cleaned[column].astype('category')


print("Categorical Feature Conversion Summary")
print("-" * 50)
print(f"Total categorical columns converted: {len(categorical_columns)}\n")


display(df_cleaned[categorical_columns].dtypes.to_frame(name="Updated Data Type"))

### 1.11 Identifying the Oral Medication Cohort

In this step, we:
1. Define and verify the presence of all 22 oral medication attributes in df_cleaned.
2. Confirm each feature records medication dosage adjustments ("No", "Steady", "Up", "Down").
3. Isolate the list of oral medication columns to support comparative efficacy analysis against Insulin therapy in Phase 3.

In [ ]:
# Step 1.11: Identify, verify, and isolate the 22 oral medication features

oral_agents = [
    "metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride",
    "acetohexamide", "glipizide", "glyburide", "tolbutamide", "pioglitazone",
    "rosiglitazone", "acarbose", "miglitol", "troglitazone", "tolazamide",
    "examide", "citoglipton", "glyburide-metformin", "glipizide-metformin",
    "glimepiride-pioglitazone", "metformin-rosiglitazone", "metformin-pioglitazone"
]

available_oral_meds = [
    col for col in oral_agents 
    if col in df_cleaned.columns
]

missing_oral_meds = [
    col for col in oral_agents 
    if col not in df_cleaned.columns
]

print("Oral Medication Cohort Audit")
print("-" * 50)
print(f"Expected Oral Medications : {len(oral_agents)}")
print(f"Identified in Dataset     : {len(available_oral_meds)}")
print(f"Missing Columns           : {len(missing_oral_meds)}")

if len(available_oral_meds) == 22:
    print("\nVerification Passed: All 22 oral medication features successfully verified.")
else:
    print(f"\nWarning: Missing medication features: {missing_oral_meds}")


print("\nDose Level Distribution for Oral Agents:")
display(df_cleaned[available_oral_meds].apply(pd.Series.value_counts).fillna(0).astype(int))

### 1.12 Final Dataset Assessment

In this step, the cleaned clinical dataset is reviewed to verify the results of the data sanitation process.

The final dataset dimensions, missing values, duplicate records, and data types are examined.

In [ ]:
# Step 1.12: Perform final verification

print("Final Dataset Shape:", df_cleaned.shape)


print("\nRemaining Missing Values:")
print(df_cleaned.isnull().sum().sum())

print("\nRemaining Duplicate Rows:", df_cleaned.duplicated().sum())


print("\nFinal Dataset Information:")
df_cleaned.info()

display(df_cleaned.head())

## Phase 2: Data Enrichment via Web Scraping

#### Objective
The diag_1 column stores each patient's main diagnosis as an ICD-9 code, like 428 or 250.8. A doctor knows that 428 means heart failure, but a hospital administrator just sees a number. In this phase I fix that by scraping the disease names from a public ICD-9 website and adding them to the dataset, so anyone reading the results can understand them.

#### Why this matters for VHN
- The CMIO asked us to explain why readmissions happen, not just report what happened. That is hard to do when the diagnoses are only numbers.
- Readable labels like "Heart failure" make the later phases (EDA, the risk score and the board report) much easier to follow.
- It also shows how an internal dataset can be improved with outside reference data.

### 2.1 Target Identification: Top 20 Primary Diagnoses

#### What I am doing
I count how often each ICD-9 code appears in diag_1 and select the 20 most frequent codes. These become the only codes that will be looked up online.

#### Why only the top 20?
1. Efficiency: The dataset contains hundreds of distinct diag_1 codes. Looking up every one would take a long time, because the scraper must wait one second between requests.
2. Ethical scraping: Fewer requests means less load on the host server and a lower risk of the IP address being blocked.
3. Pareto principle: A small number of conditions usually account for a large share of all hospital encounters. Focusing on them gives the best insight per request.
4. Project requirement: The specification explicitly asks for the top 20 codes as the scraping targets.

In [ ]:
# 2.1 Target Identification: Top 20 primary diagnosis codes (diag_1)

TOP_N = 20

top20_df = (df_cleaned['diag_1'].value_counts()
            .head(TOP_N)
            .rename_axis('diag_1')
            .reset_index(name='encounters'))
top20_df.index = top20_df.index + 1 
top20_df.index.name = 'rank'

print(f"Distinct diag_1 codes in the cleaned data: {df_cleaned['diag_1'].nunique():,}")
display(top20_df)

top20_codes = top20_df['diag_1'].tolist()
print("Scraping targets:", top20_codes)

### Findings from Step 2.1
- The most frequent primary diagnosis is 428 with 6,735 encounters, followed closely by 414 with 6,554. After these two, the counts drop quite a lot: 786 is third with 4,016 encounters.
- The top 20 mixes diabetes codes (250.8, 250.6) with other conditions that often occur alongside diabetes, such as heart-related codes (428, 414, 410, 427) and lung-related codes (486, 491).
- The counts fall steadily down the list. From the 7th code onwards they are fairly close to each other, ranging from about 1,000 to 2,300 encounters.
- Some codes are formatted differently from the others. For example, 38 has lost its leading zero (it should be 038) and V57 starts with a letter. I will need to handle this when building the scraper in step 2.2.

**Conclusion:** These 20 codes are my scraping targets for the next step.

### 2.2 Scraper Architecture

#### What I am doing
I build a scraper that looks up the description of each of the 20 target codes on the public ICD-9 website `http://icd9.chrisendres.com/`, using `requests` to download the pages and `BeautifulSoup` to read them.

#### How the scraper works
The website is organised like a book: **chapter → section → code**. For example, code `428` is inside the "Circulatory System" chapter, then inside the "Other Forms of Heart Disease (420-429)" section. The scraper starts at the main index and keeps following the link whose number range contains the code, until it finds the row for that code. For codes with a decimal part (like `250.8`), it goes one level deeper into the page of the main code (`250`).

#### Decisions I made
- Browsing instead of the search box: The site's search link did not give reliable results when I tested it, so the scraper follows the index pages instead.

- Fixing the code format: Some codes lost their leading zeros in the CSV (for example `38` should be `038`, Septicemia), so I pad numeric codes to 3 digits before looking them up. `V` and `E` codes are left as they are.

- Ethical scraping: One second pause (`time.sleep(1)`) after every request, a custom `User-Agent`, a timeout, and a maximum of 3 retries.

- Error handling: If one code fails, the scraper reports it and moves on, so one problem does not stop the whole run.

In [ ]:
# 2.2 Scraper: functions 

import re
from urllib.parse import urljoin

BASE_URL = "http://icd9.chrisendres.com/"
START_URL = BASE_URL + "index.php?action=contents"
HEADERS = {"User-Agent": "Mozilla/5.0 (Academic project - IJSE ITS2122)"}
REQUEST_DELAY = 1     
MAX_RETRIES = 3
TIMEOUT = 10


CODE_ROW = re.compile(r"^([EV]?\d{2,3}(?:\.\d+)?)\s+(.+)$")

RANGE_END = re.compile(r"\(([EV]?\d+)(?:-([EV]?\d+))?\)\s*$")

page_cache = {}      


def normalize_icd9(code):
    """Pad numeric codes to 3 digits: '38' -> '038'. E and V codes stay the same."""
    code = str(code).strip()
    if code[0] in ("E", "V"):
        return code
    whole, _, decimal = code.partition(".")
    whole = whole.zfill(3)
    return f"{whole}.{decimal}" if decimal else whole


def split_code(code):
    """'V57' -> ('V', 57), '428' -> ('', 428)"""
    letter = code[0] if code[0] in ("E", "V") else ""
    return letter, int(float(code[len(letter):]))


def in_range(base, low, high):
    """Is the base code (e.g. '428') inside a section range like 420-429?"""
    b_letter, b_num = split_code(base)
    l_letter, l_num = split_code(low)
    h_letter, h_num = split_code(high)
    return b_letter == l_letter == h_letter and l_num <= b_num <= h_num


def fetch_page(url):
    """Download a page (with retries and a delay). Returns the HTML or None."""
    if url in page_cache:
        return page_cache[url]
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            response.raise_for_status()
            time.sleep(REQUEST_DELAY)         
            page_cache[url] = response.text
            return response.text
        except requests.exceptions.RequestException as error:
            print(f"   attempt {attempt}/{MAX_RETRIES} failed: {error}")
            time.sleep(REQUEST_DELAY * attempt)
    return None


def parse_rows(html, page_url):
    """Return a list of (first line of text, link or None) for every table row."""
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for tr in soup.find_all("tr"):
        cells = tr.find_all("td", recursive=False)
        if not cells:
            continue
        cell = cells[-1]                        
        text = cell.get_text("\n", strip=True).replace("\xa0", " ")
        if not text:
            continue
        first_line = text.split("\n")[0].strip()
        link = cell.find("a", href=re.compile("action=child"))
        rows.append((first_line, urljoin(page_url, link["href"]) if link else None))
    return rows


def lookup_icd9(code):
    """Find the description of one ICD-9 code. Returns the text, or None if not found."""
    code = normalize_icd9(code)
    base = code.split(".")[0]
    url, visited = START_URL, set()

    for _ in range(8):                          
        visited.add(url)
        html = fetch_page(url)
        if html is None:
            return None
        rows = parse_rows(html, url)

        for text, link in rows:
            match = CODE_ROW.match(text)
            if match and match.group(1) == code:
                return match.group(2).strip()

        if "." in code and len(code.split(".")[1]) == 1:
            for text, link in rows:
                match = CODE_ROW.match(text)
                if match and match.group(1) == code + "0":
                    return match.group(2).strip()

        next_url = None
        for text, link in rows:
            if not link or link in visited:
                continue
            match = CODE_ROW.match(text)
            if match:
                if code.startswith(match.group(1) + "."):    
                    next_url = link
                    break
            else:
                section = RANGE_END.search(text)
                if section and in_range(base, section.group(1), section.group(2) or section.group(1)):
                    next_url = link
                    break
        if next_url is None:
            return None
        url = next_url
    return None


#  quick test with 1 code before the full run
print("Test 428   ->", lookup_icd9("428"))
print("Test 250.8 ->", lookup_icd9("250.8"))

In [ ]:
# 2.2 Scraper: run on the top 20 codes 

results = []
for i, code in enumerate(top20_codes, start=1):
    print(f"[{i}/{len(top20_codes)}] Looking up {code} ...")
    try:
        description = lookup_icd9(code)
    except Exception as error:             
        print(f"   error for {code}: {error}")
        description = None
    results.append({"diag_1": code, "icd9_description": description})

icd9_lookup = pd.DataFrame(results)

found = icd9_lookup["icd9_description"].notna().sum()
print(f"\nDescriptions found: {found} out of {len(icd9_lookup)}")

failed = icd9_lookup[icd9_lookup["icd9_description"].isna()]["diag_1"].tolist()
if failed:
    print("Codes without a description:", failed)

display(icd9_lookup)

#### Findings from Step 2.2
- The scraper found descriptions for all 20 codes.
- For example, 428 is Heart failure and 414 is Other forms of chronic ischemic heart disease.
- The scraper pauses one second after every request, so it takes a minute or two to run.

### 2.3 Integration: Adding Primary_Diagnosis_Desc

#### What I am doing
I add a new column called `Primary_Diagnosis_Desc` to the main dataset. For each encounter, the column shows the plain-English description of its primary diagnosis code (diag_1), using the descriptions I scraped in step 2.2.

#### How I label the rows
- If `diag_1` is one of the top 20 codes, it gets the scraped description (for example, `428` becomes "Heart failure").
- If `diag_1` is any other code, or is missing, it gets the label **"Not in Top 20"**, as the project specification asks.
- If a top 20 code was found in the data but the scraper could not find its description, it gets **"Description Unavailable"**. This way a scraping problem is never mixed up with a rare diagnosis.

In [ ]:
# 2.3 Integration: create Primary_Diagnosis_Desc

rows_before = len(df_cleaned)

desc_map = dict(zip(icd9_lookup['diag_1'], icd9_lookup['icd9_description']))


def label_primary_diagnosis(code):
    """Return the description for a top 20 code, otherwise 'Not in Top 20'."""
    if code not in desc_map:
        return "Not in Top 20"
    description = desc_map[code]
    return description if pd.notna(description) else "Description Unavailable"


df_cleaned['Primary_Diagnosis_Desc'] = df_cleaned['diag_1'].apply(label_primary_diagnosis)

assert len(df_cleaned) == rows_before, "Row count changed!"
assert df_cleaned['Primary_Diagnosis_Desc'].isna().sum() == 0, "Empty values found!"

print("Primary_Diagnosis_Desc added successfully.")
print(f"Dataset shape: {df_cleaned.shape}\n")
print(df_cleaned['Primary_Diagnosis_Desc'].value_counts().head(10))

df_cleaned[['encounter_id', 'diag_1', 'Primary_Diagnosis_Desc']].head(10)

#### Findings from Step 2.3
- The new column `Primary_Diagnosis_Desc` was added to the dataset. The number of rows stayed the same and there are no empty values.
- 'Not in Top 20' is the largest group with 49,274 encounters (about 49% of the data), because the top 20 codes cover only part of all the diagnoses.
- Among the described diagnoses, the most common is Heart failure with 6,735 encounters, followed by Other forms of chronic ischemic heart disease (6,554) and Symptoms involving respiratory system and other chest symptoms (4,016).

## Phase 3: Exploratory Data Analysis (EDA) and Clinical Hypotheses

#### Analytical Objectives

In this phase, Exploratory Data Analysis (EDA) is carried out to better understand the hospital readmission dataset and identify patterns that may be associated with patients returning to the hospital within 30 days. Using the cleaned dataset enriched with ICD-9 diagnostic descriptions, the analysis focuses on four main areas.

1. Readmission Distribution

2. Patient Demographics and Healthcare Differences

3. Medication Patterns and Diabetes Management

4. Hospital Stay and Discharge Analysis

### 3.1 The Readmission Landscape: Target Variable and Class Imbalance

#### Understanding the Readmission Data

Before investigating the factors that may influence hospital readmissions, it is important to understand how readmissions are distributed across the dataset. The `readmitted` variable contains three possible outcomes: `NO`, `>30`, and `<30`, representing patients who were not readmitted, readmitted after 30 days, and readmitted within 30 days, respectively.

The analysis gives particular attention to the `<30` category because the CMS Hospital Readmissions Reduction Program (HRRP) focuses on certain unplanned hospital readmissions occurring within 30 days of discharge. Hospitals may face financial penalties when their risk-adjusted readmission performance does not meet the programme's requirements.

Patients readmitted within 30 days account for approximately 11.3% of all encounters, making this the smallest category in the dataset. This imbalance is important because comparing raw encounter counts alone may hide differences in readmission patterns among patient groups.

To make these comparisons more meaningful, a new binary variable called `readmitted_30d` is created. It identifies whether an encounter resulted in a readmission within 30 days. Using this variable, readmission rates can be calculated as percentages rather than relying only on total counts. This provides a clearer basis for comparing readmission patterns across different groups.

In [ ]:
# Engineer explicit binary 30-day readmission indicator for rate modeling
df_cleaned['readmitted_30d'] = (df_cleaned['readmitted'] == '<30').astype(int)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
colors = ['#2ecc71', '#f39c12', '#e74c3c']
order = ['NO', '>30', '<30']
counts = df_cleaned['readmitted'].value_counts()[order]

# Left: Donut Chart of Proportions
wedges, texts, autotexts = ax1.pie(
    counts, labels=['Not Readmitted (NO)', 'Readmitted >30 Days', 'Readmitted <30 Days (Penalty)'],
    autopct='%1.1f%%', startangle=140, colors=colors,
    explode=(0, 0, 0.08), pctdistance=0.78,
    wedgeprops=dict(width=0.42, edgecolor='white', linewidth=2)
)
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(10)
ax1.set_title("Proportional Breakdown of Encounters", fontweight='bold', fontsize=13)

# Right: Horizontal Bar Chart of Exact Volumes
y_pos = np.arange(len(order))
bars = ax2.barh(y_pos, counts.values, color=colors, height=0.55, edgecolor='black', alpha=0.9)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(['NO (Stable)', '>30 Days (Late)', '<30 Days (HRRP Target)'], fontweight='bold')
ax2.invert_yaxis()
ax2.set_xlabel("Number of Hospital Encounters", fontsize=11)
ax2.set_title("Encounter Volume & Class Imbalance", fontweight='bold', fontsize=13)

for bar in bars:
    width = bar.get_width()
    ax2.annotate(f" {width:,} ({width/len(df_cleaned)*100:.2f}%)",
                 xy=(width, bar.get_y() + bar.get_height() / 2),
                 va='center', ha='left', fontweight='bold', fontsize=10)
ax2.set_xlim(0, counts.max() * 1.28)

plt.tight_layout()
plt.show()

### 3.2 Demographic Profiling: Age Distribution and Readmission Patterns

#### 3.2 Age as a Factor in Patient Outcomes

Age is worth examining when studying hospital readmissions among patients with Type 2 diabetes. As patients get older, they may become more vulnerable to diabetes-related complications, including macrovascular and microvascular diseases. Such complications can increase the need for hospital treatment and make recovery after discharge more challenging.

For this analysis, patient encounters are divided into 10-year age groups. The total number of encounters and the corresponding 30-day readmission rate are then compared using a dual-axis chart. The primary axis represents the number of hospital encounters, while the secondary axis displays the readmission rate as a percentage.

Looking at both measurements together helps reveal whether certain age groups have more hospital encounters and whether those groups also experience higher rates of readmission. It also helps identify age categories that may need additional attention when considering patient care and discharge planning.

In [ ]:
# Step 3.2: Dual-Axis Analysis of Age Distribution vs. 30-Day Readmission Rate
age_stats = df_cleaned.groupby('age').agg(
    Encounters=('encounter_id', 'count'),
    Readmission_Rate=('readmitted_30d', lambda x: x.mean() * 100)
).reindex(sorted(df_cleaned['age'].unique())).reset_index()

fig, ax1 = plt.subplots(figsize=(11, 5.5))

# Bar chart: Encounter Volume
bars = ax1.bar(age_stats['age'], age_stats['Encounters'], color='#3498db', alpha=0.75, width=0.6, label='Encounter Volume')
ax1.set_xlabel("Patient Age Bracket (10-Year Intervals)", fontweight='bold', fontsize=11)
ax1.set_ylabel("Number of Encounters", color='#2980b9', fontweight='bold', fontsize=11)
ax1.tick_params(axis='y', labelcolor='#2980b9')
ax1.grid(False)

for bar in bars:
    h = bar.get_height()
    ax1.annotate(f"{h:,}", (bar.get_x() + bar.get_width()/2., h + 300),
                 ha='center', va='bottom', fontsize=8.5, color='#2c3e50')

# Line chart: 30-Day Readmission Rate (%)
ax2 = ax1.twinx()
ax2.plot(age_stats['age'], age_stats['Readmission_Rate'], color='#c0392b', marker='o', linewidth=2.5, markersize=8, label='30-Day Readmission Rate (%)')
ax2.set_ylabel("30-Day Readmission Rate (%)", color='#c0392b', fontweight='bold', fontsize=11)
ax2.tick_params(axis='y', labelcolor='#c0392b')
ax2.axhline(df_cleaned['readmitted_30d'].mean() * 100, color='#7f8c8d', linestyle='--', linewidth=1.2, label='Network Avg Rate')
ax2.set_ylim(0, max(age_stats['Readmission_Rate']) * 1.25)

for i, txt in enumerate(age_stats['Readmission_Rate']):
    ax2.annotate(f"{txt:.1f}%", (age_stats['age'][i], age_stats['Readmission_Rate'][i] + 0.5),
                 ha='center', fontweight='bold', color='#c0392b', fontsize=9)

plt.title("Inpatient Cohort Volume vs. 30-Day Readmission Risk Across Age Brackets", fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

### 3.2 Intersectional Equity Analysis: Readmission by Race and Biological Sex

#### Examining Intersectional Disparities in Readmission Rates

Patient readmission patterns may differ across demographic groups, making it critical to examine whether healthcare delivery and discharge transitions achieve equitable outcomes across diverse patient populations.

As mandated by the project specification, we analyze 30-day readmission rates stratified jointly by race and biological sex. We present:
1. **Grouped Bar Chart (`Readmission by Race and Gender`):** Directly compares female and male 30-day readmission percentages within each racial category.
2. **Intersectional Risk Matrix Heatmap:** Provides the full 2D cross-tabulated risk profile compared to the network benchmark (11.34%).

This dual visualization highlights intersectional vulnerabilities—identifying where gender disparities widen within specific racial cohorts to inform targeted, culturally competent post-discharge support.

In [ ]:
# Step 3.2: Grouped Bar Chart and Intersectional Matrix of Readmission by Race and Gender
df_demo = df_cleaned[df_cleaned['race'].notna() & df_cleaned['gender'].isin(['Male', 'Female'])].copy()

# Calculate stratified readmission percentages
grouped_demo = df_demo.groupby(['race', 'gender'], observed=True)['readmitted_30d'].agg(
    Encounters='count',
    Readmitted_30d='sum',
    Readmission_Rate_Pct=lambda x: round(x.mean() * 100, 2)
).reset_index()

pivot_demo = df_demo.pivot_table(index='race', columns='gender', values='readmitted_30d', aggfunc=lambda x: x.mean() * 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5))

# Panel 1: Explicit Grouped Bar Chart as requested by project specification
sns.barplot(
    data=grouped_demo,
    x='race',
    y='Readmission_Rate_Pct',
    hue='gender',
    palette={'Male': '#2980b9', 'Female': '#e74c3c'},
    ax=ax1,
    edgecolor='black',
    alpha=0.85
)
ax1.axhline(df_cleaned['readmitted_30d'].mean() * 100, color='black', linestyle='--', linewidth=1.5, label='Network Average (11.34%)')
ax1.set_title("Readmission by Race and Gender", fontweight='bold', fontsize=13)
ax1.set_xlabel("Patient Race", fontweight='bold')
ax1.set_ylabel("30-Day Readmission Rate (%)", fontweight='bold')
ax1.set_ylim(0, 16)
ax1.legend(title='Gender', loc='upper right')
ax1.tick_params(axis='x', rotation=20)

for p in ax1.patches:
    h = p.get_height()
    if h > 0:
        ax1.annotate(f"{h:.2f}%", (p.get_x() + p.get_width() / 2., h + 0.3),
                     ha='center', va='bottom', fontsize=9, fontweight='bold')

# Panel 2: Annotated Intersectional Heatmap Matrix
sns.heatmap(pivot_demo, annot=True, fmt=".2f", cmap="YlOrRd", cbar_kws={'label': '30-Day Readmission (%)'}, ax=ax2, linewidths=1)
ax2.set_title("Intersectional Risk Matrix (Race × Gender)", fontweight='bold', fontsize=13)
ax2.set_xlabel("Biological Sex", fontweight='bold')
ax2.set_ylabel("Reported Race", fontweight='bold')

plt.tight_layout()
plt.show()

print("Grouped Demographic Readmission Summary:")
print(grouped_demo.to_string(index=False))


### 3.3 Medication Analysis: Insulin vs Antidiabetics

#### 3.3 Classifying Diabetes Treatment Approaches

The dataset contains information about insulin treatment and 22 additional oral antidiabetic medications, including `metformin`, `glipizide`, and `pioglitazone`. Examining these medications individually would make it difficult to compare overall treatment patterns. Therefore, patient encounters are divided into four medication groups based on the treatments recorded in the dataset.

1. **Insulin Monotherapy:** Encounters where insulin is recorded as the only diabetes medication, with no oral antidiabetic agents.

2. **Combination Therapy (Insulin + Oral):** Encounters where insulin is used together with at least one oral antidiabetic medication.

3. **Oral Agents Only:** Encounters where patients receive one or more oral antidiabetic medications without insulin.

4. **No Diabetes Medication:** Encounters where no diabetes medication is recorded during the hospital stay.

After assigning encounters to these groups, their 30-day readmission rates are compared to identify differences in outcomes between treatment categories.

One of the main questions explored in this analysis is whether patients receiving insulin, either alone or alongside oral medications, have higher readmission rates than those treated exclusively with oral agents. This may help reveal associations between treatment patterns and patient outcomes, although medication use alone cannot establish the severity of a patient's condition or prove that a particular treatment causes readmission.

In [ ]:
# Step 3.3: Medication Regimen Classification & Readmission Efficacy

def classify_medication_regimen(row):
    has_insulin = (row['insulin'] != 'No')
    has_oral = any(row[med] != 'No' for med in oral_agents)
    if has_insulin and has_oral: return 'Combination (Insulin + Oral)'
    elif has_insulin and not has_oral: return 'Insulin Monotherapy'
    elif not has_insulin and has_oral: return 'Oral Agents Only'
    else: return 'No Diabetes Medication'

df_cleaned['Medication_Cohort'] = df_cleaned.apply(classify_medication_regimen, axis=1)

med_summary = df_cleaned.groupby('Medication_Cohort')['readmitted_30d'].mean().mul(100).sort_values()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Left: Horizontal Lollipop Chart
y_pos = np.arange(len(med_summary))
ax1.hlines(y=y_pos, xmin=0, xmax=med_summary.values, color='#16a085', alpha=0.8, linewidth=3)
ax1.plot(med_summary.values, y_pos, "o", markersize=11, color='#e74c3c')
ax1.set_yticks(y_pos)
ax1.set_yticklabels(med_summary.index, fontweight='bold')
ax1.axvline(df_cleaned['readmitted_30d'].mean() * 100, color='black', linestyle='--', label='Network Avg')
ax1.set_xlim(0, 16)
ax1.set_xlabel("30-Day Readmission Rate (%)", fontweight='bold')
ax1.set_title("30-Day Risk Ranking by Pharmacological Regimen", fontweight='bold', fontsize=12)
for i, v in enumerate(med_summary.values):
    ax1.text(v + 0.4, i, f"{v:.2f}%", va='center', fontweight='bold', color='#c0392b')
ax1.legend(loc='lower right')

# Right: 100% Stacked Bar Chart across all readmission states
ct_med = pd.crosstab(df_cleaned['Medication_Cohort'], df_cleaned['readmitted'], normalize='index')[['NO', '>30', '<30']] * 100
ct_med.loc[med_summary.index].plot(kind='barh', stacked=True, color=['#2ecc71', '#f39c12', '#e74c3c'], ax=ax2, edgecolor='white')
ax2.set_title("Full Readmission Outcome Split by Regimen (%)", fontweight='bold', fontsize=12)
ax2.set_xlabel("Percentage of Cohort (%)", fontweight='bold')
ax2.set_ylabel("")
ax2.legend(title='Readmitted', loc='lower center', bbox_to_anchor=(0.5, -0.25), ncol=3)

plt.tight_layout()
plt.show()

### 3.5 Inpatient Medication Titration (`change`) and Glycemic Instability

#### Monitoring Changes in Diabetes Treatment

Medication adjustments during a hospital stay can provide useful information about how a patient's diabetes is being managed. In this dataset, the `change` variable indicates whether a patient's diabetes medication was modified (`Ch`) or remained unchanged (`No`) during hospitalization.

The `insulin` variable provides additional details about insulin treatment, with four possible values: `Up`, `Down`, `Steady`, and `No`. These categories indicate whether insulin dosage was increased, decreased, maintained at the same level, or not recorded as being used.

This analysis compares 30 day readmission rates across these medication adjustment categories to examine whether patients who experienced dosage changes had different readmission patterns from those whose treatment remained unchanged. Particular attention is given to upward and downward insulin adjustments, as these may reflect changes in the patient's treatment requirements during the hospital stay.

The main objective is to determine whether inpatient medication adjustments are associated with an increased likelihood of returning to the hospital within 30 days. The findings may provide insight into the relationship between diabetes management during hospitalization and patient outcomes after discharge.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

chg_regimen = df_cleaned.groupby(['Medication_Cohort', 'change'])['readmitted_30d'].mean().mul(100).reset_index()
chg_regimen['Dosage_Status'] = chg_regimen['change'].map({'Ch': 'Dosage Changed (Ch)', 'No': 'Unchanged (No)'})

sns.barplot(data=chg_regimen, x='Medication_Cohort', y='readmitted_30d', hue='Dosage_Status', palette=['#e67e22', '#34495e'], ax=ax1)
ax1.set_title("Readmission Rate by Regimen & Dosage Change", fontweight='bold', fontsize=12)
ax1.set_xlabel("Medication Cohort", fontweight='bold')
ax1.set_ylabel("30-Day Readmission Rate (%)", fontweight='bold')
ax1.tick_params(axis='x', rotation=15)
for p in ax1.patches:
    if p.get_height() > 0:
        ax1.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width()/2., p.get_height() + 0.2),
                     ha='center', fontsize=8.5, fontweight='bold')


ins_order = ['No', 'Steady', 'Up', 'Down']
ins_rates = df_cleaned.groupby('insulin')['readmitted_30d'].mean().mul(100).reindex(ins_order).reset_index()

sns.barplot(data=ins_rates, x='insulin', y='readmitted_30d', palette='Reds', ax=ax2, edgecolor='black')
ax2.set_title("Impact of Insulin Titration Direction on 30-Day Return", fontweight='bold', fontsize=12)
ax2.set_xlabel("Inpatient Insulin Dosage Status", fontweight='bold')
ax2.set_ylabel("30-Day Readmission Rate (%)", fontweight='bold')
ax2.axhline(df_cleaned['readmitted_30d'].mean() * 100, color='black', linestyle='--', label='Network Avg')
for p in ax2.patches:
    ax2.annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width()/2., p.get_height() + 0.3),
                 ha='center', fontweight='bold', fontsize=10)
ax2.legend()

plt.tight_layout()
plt.show()

### 3.6 Multicollinearity Analysis of Numerical Clinical Features

#### Examining Linear (Pearson) and Monotonic (Spearman) Relationships

Before incorporating numerical features into clinical complexity algorithms or predictive models, evaluating multicollinearity is essential. High correlation between predictor variables causes variance inflation and computational instability.

To rigorously assess multicollinearity, we compute both:
1. **Pearson Correlation ($r$):** Measures linear co-movement between clinical features.
2. **Spearman Rank Correlation ($\rho$):** Explicitly evaluates monotonic associations without assuming normal distribution or linearity, critical for skewed healthcare utilization metrics (e.g., prior emergency and inpatient visits).

The analysis investigates eight numerical clinical features: `time_in_hospital`, `num_lab_procedures`, `num_procedures`, `num_medications`, `number_outpatient`, `number_emergency`, `number_inpatient`, and `number_diagnoses` against the target outcome (`readmitted_30d`).

In [ ]:
# Step 3.6: Multicollinearity Evaluation with Explicit Pearson and Spearman Correlation Calculations
num_cols = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses', 'readmitted_30d'
]


pearson_corr = df_cleaned[num_cols].corr(method='pearson')
spearman_corr = df_cleaned[num_cols].corr(method='spearman')


print("="*80)
print("EXPLICIT SPEARMAN RANK CORRELATION COEFFICIENTS (ρ)")
print("="*80)
pairs = [
    ('time_in_hospital', 'num_lab_procedures'),
    ('time_in_hospital', 'num_medications'),
    ('num_medications', 'num_lab_procedures'),
    ('number_inpatient', 'readmitted_30d'),
    ('number_emergency', 'readmitted_30d'),
    ('number_diagnoses', 'time_in_hospital')
]
for f1, f2 in pairs:
    rho = spearman_corr.loc[f1, f2]
    r = pearson_corr.loc[f1, f2]
    print(f"  {f1:<20} vs. {f2:<20} -> Spearman ρ = {rho:+.4f} | Pearson r = {r:+.4f}")

mask = np.triu(np.ones_like(pearson_corr, dtype=bool))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7.5))

sns.heatmap(
    pearson_corr, mask=mask, annot=True, fmt='.2f',
    cmap='vlag', vmin=-0.2, vmax=0.5, center=0,
    square=True, linewidths=1.2, cbar_kws={"shrink": 0.75, "label": "Pearson Correlation (r)"}, ax=ax1
)
ax1.set_title("Pearson Linear Correlation (r)", fontweight='bold', fontsize=12)
ax1.tick_params(axis='x', rotation=35)

sns.heatmap(
    spearman_corr, mask=mask, annot=True, fmt='.2f',
    cmap='vlag', vmin=-0.2, vmax=0.5, center=0,
    square=True, linewidths=1.2, cbar_kws={"shrink": 0.75, "label": "Spearman Correlation (ρ)"}, ax=ax2
)
ax2.set_title("Spearman Rank Correlation (ρ)", fontweight='bold', fontsize=12)
ax2.tick_params(axis='x', rotation=35)

plt.suptitle("Multicollinearity Evaluation of Numerical Clinical Features", fontweight='bold', fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

print("\nMulticollinearity Verdict:")
print("All pairwise correlations remain below the collinearity threshold (|r| < 0.70; |ρ| < 0.70).")
print("Features are non-redundant and suitable for independent inclusion in scoring algorithms and models.")


### 3.7 Relationship Between Length of Stay and Laboratory Procedures

#### Investigating Hospital Stay and Testing Activity

This analysis examines whether the number of laboratory procedures performed during a hospital stay tends to increase with the length of that stay. The two variables considered are `time_in_hospital` and `num_lab_procedures`. The main objective is to identify whether a positive linear relationship exists between them.

Two charts are used to examine this relationship from different perspectives:

- **Left Chart  – Bar Chart:** Displays the average number of laboratory procedures performed for each hospital stay duration, covering stays from 1 to 14 days. This makes it easier to compare laboratory activity across different lengths of stay.

- **Right Chart – Scatter Plot with Regression Trendline:** Uses a randomly selected sample of 1,000 patient encounters to display the relationship between hospital stay duration and the number of laboratory procedures. A red regression line is added to highlight the overall direction of the relationship.

Together, these charts provide a clearer view of how laboratory testing activity varies with hospital stay duration. They also help determine whether longer hospital stays are generally associated with a greater number of laboratory procedures.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

avg_labs = df_cleaned.groupby('time_in_hospital')['num_lab_procedures'].mean().reset_index()

sns.barplot(data=avg_labs, x='time_in_hospital', y='num_lab_procedures', color='#3498db', ax=ax1)
ax1.set_title("Average Lab Procedures by Hospital Days", fontweight='bold')
ax1.set_xlabel("Time in Hospital (Days)")
ax1.set_ylabel("Average Number of Lab Tests")

for p in ax1.patches:
    ax1.annotate(f"{p.get_height():.0f}", 
                 (p.get_x() + p.get_width() / 2., p.get_height() + 0.8),
                 ha='center', fontsize=9, fontweight='bold')

sample_df = df_cleaned.sample(n=1000, random_state=42)

sns.regplot(data=sample_df, x='time_in_hospital', y='num_lab_procedures',
            scatter_kws={'alpha': 0.25, 'color': '#2c3e50'},
            line_kws={'color': 'red', 'linewidth': 2}, ax=ax2)

ax2.set_title("Linear Correlation: Hospital Stay vs. Lab Tests", fontweight='bold')
ax2.set_xlabel("Time in Hospital (Days)")
ax2.set_ylabel("Number of Lab Procedures")

plt.tight_layout()
plt.show()

corr_value = df_cleaned['time_in_hospital'].corr(df_cleaned['num_lab_procedures'])
print(f"Pearson Correlation (r): {corr_value:.2f} -> Positive Linear Correlation (Longer stays = More lab tests)")

### 3.8 Initial Length of Stay Distribution and Post-Acute Discharge Disposition

#### Evaluating Inpatient Utilization and Care Transition Risk

This section examines two critical operational metrics driving readmissions:
1. **Standalone Length of Stay (LOS) Boxplot:** As explicitly required by the project specification, we compare `time_in_hospital` grouped by readmission status (`NO` vs. `<30`). We assess medians, interquartile ranges (IQR), and extreme outlier stays to investigate if readmitted patients experienced longer initial hospitalizations.
2. **Post-Acute Discharge Disposition Analysis:** Evaluates whether patients transferred to Skilled Nursing Facilities (SNFs) or home health exhibit higher recidivism rates than patients discharged home.

In [ ]:
# Step 3.8: Standalone Length of Stay Boxplot and Discharge Disposition Analysis
def simplify_discharge_disp(desc):
    if pd.isna(desc): return 'Other Disposition'
    desc_str = str(desc).lower()
    if 'home' in desc_str and 'service' not in desc_str:
        return 'Discharged to Home'
    elif 'snf' in desc_str or 'skilled nursing' in desc_str:
        return 'Skilled Nursing Facility (SNF)'
    elif 'home health service' in desc_str:
        return 'Home with Health Service'
    elif 'short term' in desc_str or 'rehab' in desc_str or 'transfer' in desc_str:
        return 'Post-Acute / Rehab Transfer'
    else:
        return 'Other Disposition'

df_cleaned['Discharge_Group'] = df_cleaned['discharge_disposition_description'].apply(simplify_discharge_disp)
df_sub = df_cleaned[df_cleaned['readmitted'].isin(['NO', '<30'])].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5))


sns.boxplot(
    data=df_sub,
    x='readmitted',
    y='time_in_hospital',
    order=['NO', '<30'],
    palette=['#2ecc71', '#e74c3c'],
    showmeans=True,
    meanprops={'marker': 'o', 'markerfacecolor': 'yellow', 'markeredgecolor': 'black', 'markersize': 8},
    ax=ax1
)


stats_summary = df_sub.groupby('readmitted')['time_in_hospital'].describe()
for i, group in enumerate(['NO', '<30']):
    med = stats_summary.loc[group, '50%']
    mean = stats_summary.loc[group, 'mean']
    q25 = stats_summary.loc[group, '25%']
    q75 = stats_summary.loc[group, '75%']
    ax1.text(i, 13.0, f"Median: {med:.0f}d | Mean: {mean:.2f}d\nIQR: [{q25:.0f} - {q75:.0f}]d", 
             ha='center', fontsize=10, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='black', alpha=0.9))

ax1.set_title("Standalone Length of Stay Boxplot (NO vs. <30 Days)", fontweight='bold', fontsize=12)
ax1.set_xlabel("Readmission Outcome (<30 Target Cohort)", fontweight='bold')
ax1.set_ylabel("Time in Hospital (Days)", fontweight='bold')
ax1.set_ylim(0, 15)


disp_summary = df_cleaned.groupby('Discharge_Group')['readmitted_30d'].agg(
    Encounters='count',
    Rate=lambda x: x.mean() * 100
).sort_values(by='Rate', ascending=True).reset_index()

bars = ax2.barh(disp_summary['Discharge_Group'], disp_summary['Rate'], color='#8e44ad', alpha=0.85, edgecolor='black')
ax2.axvline(df_cleaned['readmitted_30d'].mean() * 100, color='red', linestyle='--', linewidth=1.5, label='Network Average (11.34%)')
ax2.set_title("30-Day Readmission Rate by Discharge Pathway", fontweight='bold', fontsize=12)
ax2.set_xlabel("30-Day Readmission Rate (%)", fontweight='bold')
ax2.set_xlim(0, 20)

for bar in bars:
    w = bar.get_width()
    ax2.annotate(f" {w:.2f}%", (w, bar.get_y() + bar.get_height()/2),
                 va='center', ha='left', fontweight='bold', fontsize=10)
ax2.legend(loc='lower right')

plt.tight_layout()
plt.show()

print("Length of Stay Detailed Statistics:")
print(stats_summary[['count', 'mean', 'std', '50%', '25%', '75%']].rename(columns={'50%': 'median'}))


### 3.9 Comorbidity Analysis: Secondary and Tertiary Clinical Diagnoses (`diag_2` & `diag_3`)

#### Contextualizing Multimorbidity in Hospitalized Diabetic Patients

In response to the Chief Medical Information Officer's mandate to provide *diagnostic clarity* and uncover the co-occurring clinical conditions that complicate diabetes management (Specification Section 2.2 & 3.2), this section investigates **secondary (`diag_2`)** and **tertiary (`diag_3`)** diagnoses.

Hospitalized diabetic patients rarely suffer from glycemic dysregulation alone. Instead, long-standing micro- and macro-vascular complications create complex multimorbid phenotypes. To transform cryptic numerical ICD-9 codes into actionable administrative and clinical intelligence, we:
1. Map ICD-9 codes to standardized clinical organ systems (Circulatory, Endocrine/Metabolic, Respiratory, Genitourinary/Renal, Neoplasms, Digestive).
2. Identify the top 10 specific secondary diagnoses that most frequently co-occur with diabetes.
3. Evaluate the empirical 30-day readmission rate across comorbidity categories to uncover high-risk clinical phenotypes.

In [ ]:
# Step 3.9: Secondary & Tertiary Comorbidity Analysis (diag_2 and diag_3)

def categorize_icd9_clinical(code):
    """Categorize ICD-9 codes into standardized clinical disease categories."""
    if pd.isna(code) or str(code).strip() in ['?', '']:
        return 'Missing/Unknown'
    code_str = str(code).strip()
    if code_str.startswith('V') or code_str.startswith('E'):
        return 'Supplementary (V/E Codes)'
    try:
        val = float(code_str)
        if (390 <= val <= 459) or val == 785:
            return 'Circulatory (Heart/Vascular)'
        elif (460 <= val <= 519) or val == 786:
            return 'Respiratory'
        elif (580 <= val <= 629) or val == 788:
            return 'Genitourinary (Renal/UTI)'
        elif 250 <= val < 251:
            return 'Diabetes Mellitus'
        elif 240 <= val <= 279:
            return 'Metabolic & Electrolyte'
        elif (520 <= val <= 579) or val == 787:
            return 'Digestive System'
        elif 800 <= val <= 999:
            return 'Injury & Poisoning'
        elif 710 <= val <= 739:
            return 'Musculoskeletal'
        elif 140 <= val <= 239:
            return 'Neoplasms'
        else:
            return 'Other Conditions'
    except:
        return 'Other Conditions'

# Apply broad categorization
df_cleaned['diag_2_category'] = df_cleaned['diag_2'].apply(categorize_icd9_clinical)
df_cleaned['diag_3_category'] = df_cleaned['diag_3'].apply(categorize_icd9_clinical)

# Clinical name dictionary for top individual ICD-9 secondary codes
top_icd9_names = {
    '276': 'Fluid & Electrolyte Disorders',
    '428': 'Congestive Heart Failure',
    '250': 'Diabetes Complications',
    '427': 'Cardiac Dysrhythmias',
    '401': 'Essential Hypertension',
    '496': 'Chronic Airway Obstruction (COPD)',
    '599': 'Urinary Tract Infection',
    '403': 'Hypertensive Chronic Kidney Disease',
    '414': 'Chronic Ischemic Heart Disease',
    '411': 'Post-Infarction Angina / Coronary',
    '585': 'Chronic Kidney Disease (CKD)',
    '272': 'Disorders of Lipoid Metabolism'
}

# Top 10 specific secondary diagnoses
top10_diag2_codes = df_cleaned['diag_2'].value_counts().head(10).reset_index()
top10_diag2_codes.columns = ['ICD9_Code', 'Encounter_Count']
top10_diag2_codes['Condition_Name'] = top10_diag2_codes['ICD9_Code'].astype(str).map(
    lambda c: top_icd9_names.get(c.split('.')[0], f'ICD-9 {c}')
)
top10_diag2_codes['Pct_of_Encounters'] = (top10_diag2_codes['Encounter_Count'] / len(df_cleaned) * 100).round(2)

# Readmission rate by secondary disease organ system
category_summary = df_cleaned.groupby('diag_2_category', observed=True)['readmitted_30d'].agg(
    Encounters='count',
    Readmissions_30d='sum',
    Readmission_Rate_Pct=lambda x: round(x.mean() * 100, 2)
).sort_values(by='Encounters', ascending=False).reset_index()


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 6))

# Plot 1: Top 10 Specific Secondary Comorbidities
sns.barplot(data=top10_diag2_codes, x='Encounter_Count', y='Condition_Name', palette='mako', ax=ax1, edgecolor='black')
ax1.set_title("Top 10 Secondary Diagnoses (diag_2) Co-Occurring with Diabetes", fontweight='bold', fontsize=12)
ax1.set_xlabel("Number of Patient Encounters", fontweight='bold')
ax1.set_ylabel("Secondary Comorbidity Condition", fontweight='bold')
for bar in ax1.patches:
    w = bar.get_width()
    ax1.annotate(f" {int(w):,} ({w/len(df_cleaned)*100:.1f}%)", 
                 (w, bar.get_y() + bar.get_height()/2),
                 va='center', ha='left', fontsize=9, fontweight='bold')
ax1.set_xlim(0, max(top10_diag2_codes['Encounter_Count']) * 1.25)

# Plot 2: 30-Day Readmission Rate across Secondary Disease Categories
cat_plot = category_summary[category_summary['Encounters'] >= 1000].sort_values(by='Readmission_Rate_Pct', ascending=True)
bars = ax2.barh(cat_plot['diag_2_category'], cat_plot['Readmission_Rate_Pct'], color='#e67e22', edgecolor='black', alpha=0.85)
ax2.axvline(df_cleaned['readmitted_30d'].mean() * 100, color='red', linestyle='--', linewidth=1.5, label='Network Average (11.34%)')
ax2.set_title("30-Day Readmission Rate by Secondary Comorbidity Category", fontweight='bold', fontsize=12)
ax2.set_xlabel("30-Day Readmission Rate (%)", fontweight='bold')
ax2.set_ylabel("Clinical Category", fontweight='bold')
ax2.set_xlim(0, 18)
for bar in bars:
    w = bar.get_width()
    ax2.annotate(f" {w:.2f}%", (w, bar.get_y() + bar.get_height()/2),
                 va='center', ha='left', fontsize=10, fontweight='bold')
ax2.legend(loc='lower right')

plt.tight_layout()
plt.show()

print("Top 10 Secondary Comorbidities Table:")
print(top10_diag2_codes[['ICD9_Code', 'Condition_Name', 'Encounter_Count', 'Pct_of_Encounters']].to_string(index=False))
print("\nSecondary Diagnosis Organ System Summary:")
print(category_summary.to_string(index=False))


#### Clinical Findings from Comorbidity Analysis (`diag_2` & `diag_3`)

1. **Cardiovascular Overload:** Over **31.3%** of secondary diagnoses (`diag_2`) . Heart failure (`ICD 428`), hypertension (`ICD 401`), and chronic ischemic heart disease (`ICD 414`) are the leading comorbid burdens.
2. **Renal & Metabolic Compromise:** Fluid and electrolyte derangements (`ICD 276`; 6,653 encounters) and hypertensive chronic kidney disease (`ICD 403` / `585`) represent major secondary drivers. Genitourinary / renal comorbidities demonstrate an elevated 30-day readmission rate of **11.96%** in `diag_2` , demonstrating that diabetic nephropathy severely increases post-discharge recidivism.
3. **High-Risk Oncology Phenotype:** Neoplasms co-occurring with diabetes, while representing a smaller volume (2,452 encounters), suffer the highest 30-day readmission rate at **15.50%**.

These findings demonstrate that hospital readmissions are heavily propelled by multisystem chronic failure, directly validating the integration of the comorbidity burden ($C$) component in the Vitality Complexity Index.

## Phase 4: Feature Engineering — The Vitality Complexity Index (VCI)

#### Theoretical & Clinical Foundation
To empower floor nurses, care coordinators, and discharge planners with a practical, bedside risk stratification tool, we translate clinical heuristics into an algorithmic risk score: the **Vitality Complexity Index (VCI)**. 

The VCI is grounded in the validated medical literature surrounding the **LACE Index** (Length of Stay, Acuity of Admission, Comorbidities, Emergency Department Visits), adapted specifically for Vitality Health Network's operational EHR infrastructure.

#### VCI Mathematical Formulation:
$$\text{VCI\_Score} = L + A + C + E$$

| Component | Clinical Concept | Dataset Feature |
| :--- | :--- | :--- |
| **$L$ — Length of Stay** | Duration of acute inpatient stay | `time_in_hospital` |
| **$A$ — Acuity of Admission** | Emergency or trauma admission urgency | `admission_type_id` |
| **$C$ — Comorbidity Burden** | Sickness burden / co-occurring diseases | `number_diagnoses` |
| **$E$ — Emergency Intensity** | High-utilizer baseline (visits in prior year) | `number_emergency` |

Where each sub-score is calculated using predefined clinical scoring thresholds:

### Step 4.1 – Calculate L (Length of Stay)
1. **$L$ — Length of Stay Score (`time_in_hospital`):**
   - $< 1$ day: $0$ points
   - $1 - 4$ days: $1$ point
   - $5 - 13$ days: $4$ points
   - $\ge 14$ days: $7$ points

In [ ]:
# Step 4.1: Calculate L

def calculate_L_points(days):
    if days < 1:
        return 0
    elif 1 <= days <= 4:
        return 1
    elif 5 <= days <= 13:
        return 4
    else:
        return 7

df_cleaned["L_Score"] = (
    df_cleaned["time_in_hospital"].apply(calculate_L_points)
)

print("Calculate L points Complete.")

### Step 4.2 – Calculate A (Acuity of Admission)
2. **$A$ — Acuity of Admission Score (`admission_type_id`):**
   - Admitted via **Emergency** (`ID 1`) or **Trauma Center** (`ID 7`): $3$ points
   - All other elective/urgent admission types: $0$ points

In [ ]:
# Step 4.2 – Calculate A

def calculate_A_points(admission_type):

    if admission_type in [1, 7]:
        return 3

    else:
        return 0

df_cleaned["A_Score"] = (
    df_cleaned["admission_type_id"].apply(calculate_A_points)
)

print("Calculate A points Complete.")

### Step 4.3 – Calculate C (Comorbidity Burden)
3. **$C$ — Comorbidity Burden Score (`number_diagnoses` proxy):**
   - $< 4$ diagnoses: $0$ points
   - $4 - 7$ diagnoses: $3$ points
   - $\ge 8$ diagnoses: $5$ points

In [ ]:
# Step 4.3 – Calculate C

def calculate_C_points(diagnoses):

    if diagnoses < 4:
        return 0

    elif diagnoses <= 7:
        return 3

    else:
        return 5

df_cleaned["C_Score"] = (
    df_cleaned["number_diagnoses"].apply(calculate_C_points)
)

print("Calculate C points Complete.")

### Step 4.4 – Calculate E (Emergency Visit Intensity)
4. **$E$ — Emergency Visit Intensity Score (`number_emergency` in prior year):**
   - $0$ visits: $0$ points
   - $1 - 4$ visits: $3$ points
   - $> 4$ visits: $5$ points

In [ ]:
# Step 4.4 – Calculate E

def calculate_E_points(visits):

    if visits == 0:
        return 0

    elif visits <= 4:
        return 3

    else:
        return 5

df_cleaned["E_Score"] = (
    df_cleaned["number_emergency"].apply(calculate_E_points)
)

print("Calculate E points Complete.")

### Step 4.5 – Calculate the final VCI score and categorize patients into three risk levels
#### Risk Strata Categorization:
- **Low Risk:** $\text{VCI} < 7$
- **Medium Risk:** $7 \le \text{VCI} \le 10$
- **High Risk:** $\text{VCI} > 10$

In [ ]:
# Step 4.5 – Calculate composite VCI Score and define risk strata

df_cleaned["VCI_Score"] = (
    df_cleaned["L_Score"] +
    df_cleaned["A_Score"] +
    df_cleaned["C_Score"] +
    df_cleaned["E_Score"]
)

def stratify_vci_risk(score):

    if score < 7:
        return "Low Risk (<7)"

    elif score <= 10:
        return "Medium Risk (7-10)"

    else:
        return "High Risk (>10)"
        

df_cleaned["VCI_Stratum"] = (
    df_cleaned["VCI_Score"].apply(stratify_vci_risk)
)

print("VCI Algorithm execution complete.")

df_cleaned[['encounter_id', 'L_Score', 'A_Score', 'C_Score', 'E_Score', 'VCI_Score', 'VCI_Stratum']].head(5)

#### Visualizing the Distribution of Composite VCI Scores

Before evaluating readmission rates, we examine the raw distribution of composite ***VCI Scores*** across all 100,114 patient encounters.

* ***Score Range:*** Scores range from ***1 to 20*** points (Mean $\approx 8.15$, Median $= 9.0$).
* ***Clinical Tiers:*** 
  * 🟢 ***Low Risk ($\text{VCI} < 7$):*** Concentrated at scores 1, 4, and 6.
  * 🟠 ***Medium Risk ($7 \le \text{VCI} \le 10$):*** Peak volume concentrated at scores 7, 9, and 10.
  * 🔴 ***High Risk ($\text{VCI} > 10$):*** High-acuity patients with scores 11 to 20, peaking at score 12.

In [ ]:
# Visualize the distribution of composite VCI Scores

vci_dist = df_cleaned['VCI_Score'].value_counts().sort_index().reset_index()
vci_dist.columns = ['VCI_Score', 'Encounters']
vci_dist['Risk_Stratum'] = vci_dist['VCI_Score'].apply(stratify_vci_risk)

tier_palette = {
    'Low Risk (<7)': '#2ecc71',      
    'Medium Risk (7-10)': '#f39c12', 
    'High Risk (>10)': '#e74c3c'     
}


plt.figure(figsize=(14, 5))
ax = sns.barplot(
    data=vci_dist, 
    x='VCI_Score', 
    y='Encounters', 
    hue='Risk_Stratum', 
    palette=tier_palette, 
    dodge=False
)

plt.title("Distribution of Inpatient Encounters Across Vitality Complexity Index (VCI) Scores", 
          fontsize=13, fontweight='bold', pad=15)

plt.xlabel("Composite VCI Score (Points)", fontsize=11)
plt.ylabel("Number of Inpatient Encounters", fontsize=11)

plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.axvline(x=vci_dist['VCI_Score'].tolist().index(7) - 0.5, color='gray', linestyle=':', linewidth=1.5)
plt.axvline(x=vci_dist['VCI_Score'].tolist().index(11) - 0.5, color='gray', linestyle=':', linewidth=1.5)

mean_score = df_cleaned['VCI_Score'].mean()
median_score = df_cleaned['VCI_Score'].median()

plt.text(0.72, 0.85, f"VHN Population Metrics:\n• Mean VCI: {mean_score:.2f}\n• Median VCI: {median_score:.0f}", 
         transform=ax.transAxes, fontsize=10, 
         bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="#bdc3c7", alpha=0.9))
plt.legend(title="VCI Risk Tier", loc='upper left')

plt.tight_layout()
plt.show()

### 4.6 Algorithmic Validation & Empirical Risk Discrimination
To evaluate the VCI stratification, we calculate the empirical 30-day readmission rate (<30) across the three risk strata (Low, Medium, and High). We examine whether the readmission rate increases monotonically as patients advance to higher risk tiers.

In [ ]:
# Step 4.6: Empirical Validation of the VCI Stratification

vci_validation = df_cleaned.groupby('VCI_Stratum')['readmitted_30d'].agg(
    Cohort_Encounters='count',
    Readmitted_30d_Count='sum',
    Readmission_Rate_Pct=lambda x: (x.mean() * 100).round(2)
).reset_index()

order_map = {'Low Risk (<7)': 1, 'Medium Risk (7-10)': 2, 'High Risk (>10)': 3}
vci_validation['Order'] = vci_validation['VCI_Stratum'].map(order_map)
vci_validation.sort_values(by='Order', inplace=True)
vci_validation.drop(columns=['Order'], inplace=True)


print("\nEmpirical VCI Validation Results")
print("-"*100,"\n")
print(vci_validation.to_string(index=False))

In [ ]:
# Visualization: VCI Stratum Risk Discrimination
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Plot A
sns.barplot(data=vci_validation, x='VCI_Stratum', y='Cohort_Encounters', palette='Blues_d', ax=ax1)
ax1.set_title("VHN Patient Population Stratification by VCI Tier", fontweight='bold')
ax1.set_xlabel("VCI Risk Stratum")
ax1.set_ylabel("Number of Encounters")


for p in ax1.patches:
    ax1.annotate(f"{int(p.get_height()):,}\n({p.get_height()/len(df_cleaned)*100:.1f}%)", 
                 (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                 ha='center', va='center', color='white', fontweight='bold', fontsize=11)

# Plot B
sns.barplot(data=vci_validation, x='VCI_Stratum', y='Readmission_Rate_Pct', palette=['#2ecc71', '#f39c12', '#e74c3c'], ax=ax2)
ax2.set_title("Empirical 30-Day Readmission Rate (<30) by VCI Risk Stratum", fontweight='bold')
ax2.set_xlabel("VCI Risk Stratum")
ax2.set_ylabel("30-Day Readmission Rate (%)")
ax2.axhline(df_cleaned['readmitted_30d'].mean() * 100, color='black', linestyle='--', label='Network Average (11.34%)')
ax2.legend()


for p in ax2.patches:
    ax2.annotate(f"{p.get_height():.2f}%", 
                 (p.get_x() + p.get_width() / 2., p.get_height() + 0.4),
                 ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

## Project Synthesis & Final Conclusion: Vitality Health Network (VHN)

This project delivers an end-to-end data pipeline to analyze 30-day diabetic readmissions and reduce CMS HRRP penalty exposure.

---

###  Phased Analytical Milestones

* Phase 1 (Data Sanitation): Cleaned 100,114 valid encounters (from 101,766 raw records) after removing 1,652 deceased patients and dropping `weight` (96.86% missing).

* Phase 2 (Web Scraping): Scraped plain-English names for the top 20 ICD-9 codes with 100% success, confirming heart failure and ischemic heart disease as the top diabetic complications.

* Phase 3 (EDA): Established an *11.34% baseline readmission rate*. Identified three key high-risk drivers:
  * Insulin therapy: 13.24% vs. 10.53% for oral drugs.
  * Medication adjustments: 11.97% vs. 10.80% for stable doses.
  * Skilled Nursing Facility (SNF) discharge: 14.66% vs. 9.30% for home.

* Phase 4 (VCI Scoring): Built a 4-pillar, LACE-inspired score ($L + A + C + E$). Validated a clear monotonic risk progression:
  * Low Risk (<7): 8.81%
  * Medium Risk (7–10): 11.41%
  * High Risk (>10): 14.73%
  
  *(High-risk patients show a 1.67x higher readmission rate than low-risk patients).*
